## 0. Setup

In [ ]:
!pip install -q neuralforecast wandb


In [ ]:
import glob, os, time
import numpy as np
import pandas as pd

from neuralforecast import NeuralForecast

import wandb

WANDB_PROJECT = "walmart-sales-forecasting"
MODEL_NAME = "NBEATS"                              # საუკეთესო, Model Registry-ში დარეგისტრირებული მოდელი
REGISTRY_TARGET = f"wandb-registry-model/{MODEL_NAME}"
wandb.login()

_train_matches = glob.glob("/kaggle/input/**/train.csv*", recursive=True)
if _train_matches:
    DATA_DIR = os.path.dirname(_train_matches[0])
else:
    DATA_DIR = "data/raw"                          # ლოკალურად, repo root-იდან გაშვებისას
print("DATA_DIR ->", DATA_DIR)

HORIZON = 39   # test.csv-ის კვირების რაოდენობა (2012-11-02 -> 2013-07-26) -- იგივე, რაც nbeats.ipynb-ში


## 1. საუკეთესო Pipeline-ის ჩამოტვირთვა Model Registry-დან

`nbeats.ipynb`-ის Deploy Refit ეტაპმა (სექცია 7) `NeuralForecast` ობიექტი
(`nf_deploy`, გავარჯიშებული მთელ ხელმისაწვდომ ისტორიაზე, `Y_train + Y_valid`)
შეინახა `nf.save()`-ით და დარეგისტრირა Wandb Model Registry-ში, როგორც
`wandb-registry-model/NBEATS`. აქ ამ ზუსტად იმავე არტეფაქტს ვწერთ, ხელახლა
ვარჯიშის გარეშე — `run.use_artifact(...)` ჩამოტვირთავს ბოლო ვერსიას, ხოლო
`NeuralForecast.load(...)` აღადგენს გავარჯიშებულ მოდელს.

In [ ]:
run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type="inference", name=f"{MODEL_NAME}_Inference")

artifact = run.use_artifact(f"{REGISTRY_TARGET}:latest", type="model")
PIPELINE_DIR = artifact.download()
print("pipeline ჩამოტვირთულია:", PIPELINE_DIR)

nf = NeuralForecast.load(path=PIPELINE_DIR)
print("ჩატვირთული მოდელ(ებ)ი:", [type(m).__name__ for m in nf.models])


## 2. იგივე გასუფთავება/feature engineering, რაც ტრენინგისას

`NeuralForecast.save()` მხოლოდ მოდელის წონებსა და ჰიპერპარამეტრებს ინახავს —
არა Python-ის `NBEATSPipeline` wrapper-კლასს და არც fallback-ისთვის საჭირო
ისტორიულ საშუალოებს. ამიტომ ეს ორი აქ ხელახლა განისაზღვრება, **ზუსტად ისე,
როგორც `nbeats.ipynb`-ში იყო** — რომ inference-ის ქცევა ტრენინგის დროინდელს
ემთხვეოდეს. ტრენინგი აქ არ ხდება, მხოლოდ raw train.csv-ს ვკითხულობთ, რომ
`NBEATSPipeline`-ის fallback-ცხრილები (Store-Dept / Store / გლობალური საშუალო)
ავაშენოთ.

In [ ]:
train_raw = pd.read_csv(glob.glob(f"{DATA_DIR}/train.csv*")[0], parse_dates=["Date"])
stores = pd.read_csv(glob.glob(f"{DATA_DIR}/stores.csv*")[0])
features = pd.read_csv(glob.glob(f"{DATA_DIR}/features.csv*")[0], parse_dates=["Date"])

def merge_sources(df):
    feat = features.drop(columns=["IsHoliday"])  # features.csv-ს დუბლირებული IsHoliday აქვს
    return df.merge(stores, on="Store", how="left").merge(feat, on=["Store", "Date"], how="left")

def clean_data(df, is_train=True):
    df = df.copy()
    df = df.drop_duplicates()

    markdown_cols = [c for c in df.columns if c.startswith("MarkDown")]
    df[markdown_cols] = df[markdown_cols].fillna(0).clip(lower=0)

    df = df.sort_values(["Store", "Date"])
    for col in ["CPI", "Unemployment"]:
        df[col] = df.groupby("Store")[col].ffill().bfill()

    if is_train:
        df["Weekly_Sales"] = df["Weekly_Sales"].clip(lower=0)

    df["IsHoliday"] = df["IsHoliday"].astype(bool)
    df["Store"] = df["Store"].astype(int)
    df["Dept"] = df["Dept"].astype(int)
    return df

def engineer_features(df):
    df = df.copy()
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

    superbowl = pd.to_datetime(["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"])
    labor_day = pd.to_datetime(["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"])
    thanksgiving = pd.to_datetime(["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"])
    christmas = pd.to_datetime(["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"])
    df["IsSuperBowl"] = df["Date"].isin(superbowl)
    df["IsLaborDay"] = df["Date"].isin(labor_day)
    df["IsThanksgiving"] = df["Date"].isin(thanksgiving)
    df["IsChristmas"] = df["Date"].isin(christmas)

    df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
    df["ds"] = df["Date"]
    return df

train_clean = clean_data(merge_sources(train_raw), is_train=True)
train_fe = engineer_features(train_clean)
train_fe["y"] = train_fe["Weekly_Sales"]
print("history rows:", len(train_fe), "| series:", train_fe["unique_id"].nunique())


## 3. NBEATSPipeline (იგივე wrapper, რაც deployment-ზე გამოიყენებოდა)

raw dataframe -> `clean_data` -> `engineer_features` -> NBEATS-ის პროგნოზი.
საბაზისო NBEATS-ს გარე ცვლადები არ სჭირდება, ამიტომ `predict()`-საც არ
სჭირდება `futr_df` — საკმარისია `(unique_id, ds)` წყვილები, რომ პროგნოზი
`test.csv`-ის რეალურ თარიღებს დაემთხვეს. სერიები, რომლებიც NBEATS-ისთვის
უცნობია (ტრენინგისას `valid_ids`-ის ისტორიის ფილტრით გამორიცხული), ივსება
Store-Dept ისტორიული საშუალოთი, ან Store საშუალოთი, ან გლობალური საშუალოთი.

In [ ]:
class NBEATSPipeline:

    def __init__(self, nf, history_df):
        self.nf = nf
        self._dept_mean = history_df.groupby(["Store", "Dept"])["y"].mean()
        self._store_mean = history_df.groupby("Store")["y"].mean()
        self._global_mean = float(history_df["y"].mean())

    def _fallback_value(self, store, dept):
        key = (store, dept)
        if key in self._dept_mean.index:
            return float(self._dept_mean[key])
        if store in self._store_mean.index:
            return float(self._store_mean[store])
        return self._global_mean

    def predict(self, raw_df: pd.DataFrame) -> pd.DataFrame:
        df = raw_df.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df = clean_data(merge_sources(df), is_train=False)
        df = engineer_features(df)

        preds = self.nf.predict()
        model_col = [c for c in preds.columns if c not in ("unique_id", "ds")][0]
        out = df[["unique_id", "ds", "Store", "Dept"]].merge(preds, on=["unique_id", "ds"], how="left")
        out = out.rename(columns={model_col: "Weekly_Sales_Pred"})

        missing = out["Weekly_Sales_Pred"].isna()
        n_missing = int(missing.sum())
        if n_missing:
            out.loc[missing, "Weekly_Sales_Pred"] = out.loc[missing].apply(
                lambda r: self._fallback_value(r["Store"], r["Dept"]), axis=1
            )
            print(f"NBEATSPipeline: filled {n_missing}/{len(out)} rows "
                  f"({n_missing / len(out):.1%}) via Store/Dept fallback mean "
                  f"(series not seen by NBEATS due to valid_ids history filter).")

        return out.drop(columns=["Store", "Dept"])


pipeline = NBEATSPipeline(nf, history_df=train_fe)


## 4. პროგნოზი raw `test.csv`-ზე

In [ ]:
test_raw = pd.read_csv(glob.glob(f"{DATA_DIR}/test.csv*")[0], parse_dates=["Date"])
preds = pipeline.predict(test_raw)   # რეალური, დაუმუშავებელი test.csv
print(preds.shape)
preds.head()


## 5. Kaggle submission ფორმატში დაკონვერტირება

Kaggle-ს სჭირდება ორსვეტიანი CSV, `Id` (`{Store}_{Dept}_{Date}`) და
`Weekly_Sales`, ზუსტად `sampleSubmission.csv`-ის row-order-ით.

In [ ]:
submission = preds.copy()
submission["Id"] = submission["unique_id"] + "_" + submission["ds"].dt.strftime("%Y-%m-%d")
submission = submission.rename(columns={"Weekly_Sales_Pred": "Weekly_Sales"})[["Id", "Weekly_Sales"]]

sample_sub = pd.read_csv(glob.glob(f"{DATA_DIR}/sampleSubmission.csv*")[0])
missing_ids = set(sample_sub["Id"]) - set(submission["Id"])
assert not missing_ids, f"{len(missing_ids)} Id აკლია submission-ს, მაგ: {list(missing_ids)[:5]}"
submission = submission.set_index("Id").loc[sample_sub["Id"]].reset_index()

SUBMISSION_PATH = f"./submission_{MODEL_NAME.lower()}.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
print("submission შენახულია:", SUBMISSION_PATH, "| rows:", len(submission))
submission.head()


## 6. Submission-ის დალოგვა Wandb-ზე

submission ფაილი ინახება artifact-ის სახით, იმავე `NBEATS` run-ის ჯგუფში,
რომ inference-ის ყოველი გაშვება ტრასირებადი იყოს (რომელი pipeline ვერსიით
დაგენერირდა რომელი submission).

In [ ]:
sub_artifact = wandb.Artifact(name=f"{MODEL_NAME}_submission", type="submission",
                               metadata={"pipeline_artifact": artifact.name, "n_rows": len(submission)})
sub_artifact.add_file(SUBMISSION_PATH)
run.log_artifact(sub_artifact)
wandb.log({"n_rows": len(submission)})
run.finish()
print("submission artifact დალოგილია.")
